In [ ]:
!kaggle datasets download -d alizahidraja/quran-nlp
!unzip /content/quran-nlp.zip



In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import re

In [ ]:
df=pd.read_csv("/content/data/hadith/kaggle_hadiths_clean.csv")
df.info()

In [ ]:
def preprocess_text(text):
  if text=="" or text=="Nan" or pd.isna(text):
    return ""
  text = re.sub(r"[``“’”‘ﷺ]", '', text).strip()
  txt=text.replace("\u200f",'').strip()
  clean_text = re.sub(r'\s+', ' ', txt).strip()
  cleaned_text = re.sub(r"""[`"']""", '', clean_text)
  cleaned_text = re.sub(r'\s+', ' ', cleaned_text)
  return cleaned_text

df['clean_text']=df['text_ar'].apply(preprocess_text)
np_data=df['clean_text'].to_numpy()

In [ ]:
from sentence_transformers import SentenceTransformer

model =SentenceTransformer("Omartificial-Intelligence-Space/GATE-AraBert-v1",device="cuda")


In [ ]:
embeddings_list=[]

In [ ]:
for i in np_data:
  embeddings_list.append(model.encode(i))

In [ ]:
len(embeddings_list)

In [ ]:
import pickle
with open("embeddings.pkl","wb") as f:
  pickle.dump(embeddings_list,f)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def get_nearst_hadith(hadith,h_vectors,n=3):
  score_val=[]
  sent_vec=model.encode(hadith).reshape(1,-1)
  for sent_vec2 in h_vectors:
    sent_vec2=sent_vec2.reshape(1,-1)
    value=cosine_similarity(sent_vec,sent_vec2)
    score_val.append(value[0][0])
  score_val2=np.array(score_val)
  indices = np.argsort(score_val2)[-n:]
  largest_three = indices[::-1]
  scores=score_val2[[largest_three]]
  return largest_three,scores



In [ ]:
# test2="عن مَعقلِ بنِ يسارٍ قال: كانت أختُه تحتَ رجلٍ فطلَّقها ثمَّ خلَّى عنها حتَّى انقَضَت عدَّتُها ثمَّ قرَّب يخطُبُها فحمِي معقلٌ مِن ذلك وقال: خلَّى عنها وهو يقدِرُ عليها فحال بينَه وبينَها فأنزَل اللهُ: { وَإِذَا طَلَّقْتُمُ النِّسَاءَ فَبَلَغْنَ أَجَلَهُنَّ فَلَا تَعْضُلُوهُنَّ أَنْ يَنْكِحْنَ أَزْوَاجَهُنَّ إِذَا تَرَاضَوْا بَيْنَهُمْ بِالْمَعْرُوف } (البقرة: 232) »"
# test2="جاء الرسول عليه الصلاة والسلام من جهة بئر، فقابله شخص وألقى عليه التحية،ولكنه لم يجب"

# test2="نهى النبي عن المتعة في غزوة خيبر"
# test2="المسيخ الدجال"
test2="اصل اي فعل النية"

In [ ]:
get_nearst_hadith(test2,embeddings_list,4)

In [ ]:
np_data[4892]